# Campionamento audio e teorema di Nyquist

**Argomento collegato:** Digitalizzazione del suono (Lezione 11 - teoria)

## Obiettivi della lezione

In questo laboratorio esploreremo in modo pratico e **sperimentale** il teorema del campionamento (Nyquist-Shannon):

$$f_s \geq 2 \cdot f_{max}$$

dove $f_s$ è la frequenza di campionamento e $f_{max}$ è la componente in frequenza più alta presente nel segnale.

Genereremo segnali sinusoidali puri a diverse frequenze e li campioneremo:
- **sotto** la soglia di Nyquist (undersampling) → osserveremo il fenomeno dell'**aliasing**
- **esattamente alla** soglia di Nyquist (caso limite)
- **sopra** la soglia di Nyquist (campionamento corretto)

Ascolteremo e visualizzeremo gli effetti in ciascun caso, confrontando le previsioni teoriche con l'output sperimentale reale.

**Strumenti:** Python, NumPy, SciPy, sounddevice, matplotlib

## 0. Richiami teorici: dal segnale analogico al segnale digitale

Prima di scrivere codice, ripassiamo i passaggi che portano un suono (fenomeno fisico continuo) a diventare una sequenza di numeri manipolabile da un computer. Il processo di **digitalizzazione** di un segnale audio si compone di due fasi distinte:

1. **Campionamento** (asse del tempo): il segnale continuo $x(t)$ viene misurato solo in istanti discreti, separati da un intervallo di tempo costante $T_s = 1/f_s$. Il risultato è una sequenza di valori $x[n] = x(n \cdot T_s)$.
2. **Quantizzazione** (asse dell'ampiezza): ogni valore campionato, che in teoria potrebbe assumere infiniti valori reali, viene approssimato al livello discreto più vicino tra un insieme finito (determinato dal numero di bit, es. 16 bit → 65536 livelli).

In questo laboratorio ci concentriamo **esclusivamente sul campionamento** (punto 1); la quantizzazione non viene simulata esplicitamente, dato che lavoriamo con array in virgola mobile a precisione elevata (`float64`), che nella pratica approssimano bene un segnale "non quantizzato".

### Perché il campionamento può introdurre errori: l'aliasing

Il **teorema del campionamento di Nyquist-Shannon** stabilisce che un segnale a banda limitata, la cui componente in frequenza più alta è $f_{max}$, può essere ricostruito **esattamente** dai suoi campioni se e solo se:

$$f_s \geq 2 \cdot f_{max}$$

La frequenza $f_{Nyquist} = f_s / 2$ si chiama **frequenza di Nyquist**: è la frequenza massima rappresentabile correttamente per una data $f_s$.

Se questa condizione **non è rispettata**, si verifica il fenomeno dell'**aliasing**: le componenti in frequenza superiori a $f_{Nyquist}$ non spariscono, ma vengono erroneamente "ripiegate" (in inglese *folded* o *aliased*) e appaiono come componenti a frequenze più basse, indistinguibili da un segnale realmente a bassa frequenza. Una volta campionato, il segnale alias è **indistinguibile** dal segnale originale a bassa frequenza: l'informazione sulla frequenza vera è persa per sempre.

Un modo intuitivo di pensare all'aliasing: immaginate una ruota di un'auto ripresa da una videocamera. Se la ruota gira molto velocemente rispetto al frame rate della videocamera, nel video può sembrare che giri lentamente, o addirittura all'indietro. Lo stesso principio di "sotto-campionamento temporale" si applica identicamente al suono.

### Formula della frequenza di alias

Dato un segnale a frequenza $f$ campionato a $f_s < 2f$, la frequenza apparente (di alias) osservata dopo il campionamento è:

$$f_{alias} = \left| f - k \cdot f_s \right|$$

dove $k$ è l'intero più vicino a $f / f_s$ (cioè quello che porta il risultato nell'intervallo $[0, f_s/2]$).

### Perché in pratica si usa un margine

Il caso limite $f_s = 2f_{max}$ esatto è teoricamente sufficiente, ma **fragile** nella pratica: dipende criticamente dalla fase del segnale rispetto agli istanti di campionamento, e piccoli errori numerici o di temporizzazione possono causare perdita di informazione. Per questo, nei sistemi audio reali si usa sempre un margine (es. lo standard CD campiona la voce/musica, con banda utile fino a circa 20 kHz, a $f_s = 44100$ Hz, ben oltre il doppio di 20 kHz) e si applicano **filtri anti-aliasing** (filtri passa-basso analogici) prima della conversione A/D, per eliminare le componenti sopra la frequenza di Nyquist prima ancora di campionare.

### Cosa verificheremo sperimentalmente oggi

| Regime | Relazione fs vs f | Cosa aspettarsi |
|---|---|---|
| Undersampling | $f_s < 2f$ | Aliasing: frequenza percepita diversa da quella reale |
| Caso limite | $f_s = 2f$ | Risultato instabile, dipendente dalla fase |
| Campionamento corretto | $f_s > 2f$ (con margine) | Ricostruzione fedele del segnale |

## 1. Setup dell'ambiente

In [ ]:
# Librerie numeriche e per il calcolo del segnale
import numpy as np
from scipy import signal

# Libreria per la visualizzazione dei grafici
import matplotlib.pyplot as plt

# Libreria per la riproduzione audio in tempo reale
import sounddevice as sd

# Frequenza di campionamento "di riferimento" usata per la riproduzione audio.
# 44100 Hz e' lo standard storico della qualita' audio CD: viene scelta ben oltre
# il doppio della banda udibile (circa 20 kHz) per lasciare margine ai filtri
# anti-aliasing reali (che non sono "ideali" e hanno una transizione graduale).
FS_RIPRODUZIONE = 44100  # Hz

# Direttiva magica di Jupyter: mostra i grafici matplotlib incorporati
# direttamente nell'output della cella, invece che in una finestra separata.
%matplotlib inline

## Introduzione alla libreria `sounddevice`

In questo laboratorio usiamo la libreria **`sounddevice`** per riprodurre in tempo reale, dagli altoparlanti del computer, i segnali audio generati come array NumPy. E' il ponte tra i numeri che manipoliamo in Python e il suono che possiamo effettivamente ascoltare.

### Perche' serve

Un segnale audio in questo notebook e' semplicemente un array NumPy di numeri in virgola mobile (ampiezze campionate nel tempo). Questi numeri, da soli, non producono alcun suono: serve un livello software che li invii alla scheda audio del sistema operativo, la quale a sua volta li converte in un segnale elettrico analogico che pilota gli altoparlanti o le cuffie. `sounddevice` fornisce esattamente questa interfaccia, in modo semplice e diretto da Python.

### Le due funzioni principali che useremo

- **`sd.play(segnale, samplerate=fs)`**: avvia la riproduzione di `segnale` (un array NumPy) alla frequenza di campionamento `fs` indicata. La chiamata e' **non bloccante**: il codice Python prosegue subito all'istruzione successiva mentre l'audio suona in background.
- **`sd.wait()`**: blocca l'esecuzione finche' la riproduzione avviata dall'ultima `sd.play()` non e' terminata. Nel notebook la usiamo sempre subito dopo `sd.play()`, per evitare che una cella successiva parta (o che il notebook finisca di eseguire) prima che l'ascolto sia completo.

Il parametro `samplerate` e' particolarmente importante per i nostri esperimenti: e' il valore che dice alla scheda audio *a quale velocita'* leggere i campioni dell'array. Usare un `samplerate` diverso da quello con cui il segnale e' stato effettivamente generato altererebbe la frequenza percepita del suono (un effetto simile, concettualmente, a rallentare o accelerare un nastro registrato) — per questo nel notebook passiamo sempre a `sd.play` la stessa frequenza usata per costruire il segnale.

### Un esempio minimo

```python
import numpy as np
import sounddevice as sd

fs = 44100
t = np.linspace(0, 1.0, fs, endpoint=False)  # 1 secondo di campioni
segnale = 0.5 * np.sin(2 * np.pi * 440 * t)  # tono puro a 440 Hz

sd.play(segnale, samplerate=fs)
sd.wait()  # attende che il suono finisca prima di proseguire
```

### Alcune avvertenze pratiche

- **Volume:** l'ampiezza del segnale (valori tra -1 e 1) corrisponde al volume percepito. Ampiezze vicine o superiori a 1 possono causare distorsione (*clipping*): per questo nelle funzioni del notebook usiamo valori di default moderati (es. 0.5).
- **Dispositivo audio:** `sounddevice` riproduce sul dispositivo audio di output predefinito del sistema operativo. Se non sentite nulla, verificate che il volume del computer non sia azzerato e che il dispositivo corretto (altoparlanti o cuffie) sia selezionato come output predefinito.
- **Blocco delle celle:** senza `sd.wait()`, l'esecuzione delle celle successive potrebbe procedere mentre l'audio sta ancora suonando, rendendo piu' difficile seguire gli esperimenti in ordine — per questo la coppia `sd.play` + `sd.wait` viene usata sistematicamente in tutto il notebook.

## 2. Generazione di segnali sinusoidali

Un tono puro a frequenza $f$ è descritto dall'equazione:

$$x(t) = A \sin(2\pi f t + \varphi)$$

dove:
- $A$ è l'ampiezza (legata al volume percepito)
- $f$ è la frequenza in Hz (legata all'altezza/pitch percepito)
- $\varphi$ è la fase iniziale in radianti (useremo questo parametro più avanti per esplorare il caso limite di Nyquist)

Costruiamo una funzione che genera un tono puro campionato a una data frequenza di campionamento, così da poterlo sia visualizzare sia riprodurre.

In [ ]:
def genera_tono(freq, durata=2.0, fs=FS_RIPRODUZIONE, ampiezza=0.5, fase=0.0):
    """Genera un tono sinusoidale puro campionato.

    Parametri
    ---------
    freq : float
        Frequenza del segnale in Hz (l'informazione che vogliamo "catturare"
        correttamente tramite il campionamento).
    durata : float
        Durata del segnale in secondi.
    fs : int
        Frequenza di campionamento in Hz: quante misurazioni al secondo
        effettuiamo del segnale continuo.
    ampiezza : float
        Ampiezza del segnale, tra 0 e 1 (0 = silenzio, 1 = volume massimo
        senza distorsione).
    fase : float
        Fase iniziale in radianti. Utile piu' avanti per mostrare come,
        nel caso limite fs = 2f, il risultato del campionamento dipenda
        criticamente dalla fase.

    Ritorna
    -------
    segnale : np.ndarray
        Array dei valori campionati del segnale.
    t : np.ndarray
        Array degli istanti di tempo corrispondenti (in secondi).
    """
    # Costruiamo il vettore degli istanti di campionamento: da 0 a "durata",
    # con un numero di punti pari a fs * durata (una misura ogni 1/fs secondi).
    # endpoint=False evita di includere l'istante finale, per avere esattamente
    # fs campioni per ogni secondo di segnale.
    t = np.linspace(0, durata, int(fs * durata), endpoint=False)

    # Valutiamo la sinusoide negli istanti campionati: e' qui che avviene,
    # concettualmente, il "campionamento" del segnale continuo x(t) = A*sin(2*pi*f*t + phi)
    segnale = ampiezza * np.sin(2 * np.pi * freq * t + fase)

    return segnale, t


def plot_forma_onda(segnale, t, fs, n_ms=10, titolo=""):
    """Visualizza i primi n_ms millisecondi della forma d'onda.

    Mostrare solo una piccola finestra iniziale (pochi millisecondi) e' importante:
    su una durata di 1-2 secondi le singole oscillazioni sarebbero troppo fitte
    per essere distinguibili visivamente.
    """
    # Numero di campioni corrispondenti a n_ms millisecondi, dato fs
    n_campioni = int(fs * n_ms / 1000)

    plt.figure(figsize=(8, 3))
    # marker='o' mostra esplicitamente i singoli campioni discreti (non solo
    # la linea continua interpolata), per rendere visibile la natura discreta
    # del segnale campionato
    plt.plot(t[:n_campioni] * 1000, segnale[:n_campioni], marker='o', markersize=3)
    plt.xlabel("Tempo (ms)")
    plt.ylabel("Ampiezza")
    plt.title(titolo)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

Generiamo e ascoltiamo tre toni a frequenze diverse: 440 Hz (nota La, riferimento standard di accordatura), 1000 Hz, 4000 Hz.

In [ ]:
frequenze_test = [440, 1000, 4000]

for f in frequenze_test:
    # Generiamo il segnale campionato correttamente (fs molto maggiore di 2f,
    # quindi nessun rischio di aliasing in questa fase)
    seg, t = genera_tono(f, durata=1.5)

    # Visualizziamo solo i primi 10 ms per poter distinguere le singole oscillazioni
    plot_forma_onda(seg, t, FS_RIPRODUZIONE, n_ms=10, titolo=f"Tono a {f} Hz (primi 10 ms)")

    # Riproduciamo il suono: sd.play e' non bloccante, sd.wait() attende la fine
    print(f"Riproduzione tono a {f} Hz...")
    sd.play(seg, samplerate=FS_RIPRODUZIONE)
    sd.wait()

**Domanda per la discussione in aula:** cosa cambia, ad orecchio, tra i tre toni? Cosa cambia nella forma d'onda visualizzata?

> Nota empirica: osservate i valori reali riportati dai grafici (numero di oscillazioni in 10 ms) e verificate che corrispondano a quanto atteso dalla frequenza — non date per scontato il risultato teorico senza controllarlo sul grafico effettivo.

### Esercizio 2.1 (livello base)

Calcolate a mano quante oscillazioni complete dovreste osservare in 10 ms per un tono a 440 Hz, 1000 Hz e 4000 Hz. Confrontate il risultato con quanto effettivamente visibile nei grafici sopra.

*Suggerimento:* in $T$ secondi, un segnale a frequenza $f$ compie $f \cdot T$ oscillazioni complete.

### Esercizio 2.2 (livello base)

Modificate la funzione `genera_tono` (o richiamatela con parametri diversi) per generare un tono a 220 Hz e uno a 880 Hz. Cosa notate rispetto al tono di riferimento a 440 Hz? (Suggerimento: il rapporto tra le frequenze è collegato al concetto musicale di ottava.)

## 3. Campionamento sotto la soglia di Nyquist: l'aliasing

Se campioniamo un segnale a frequenza $f$ con una frequenza di campionamento $f_s < 2f$, il segnale ricostruito a partire dai campioni **non rappresenta correttamente** il segnale originale: appare invece a una frequenza diversa, detta **frequenza di alias**.

### Perché accade: un'intuizione geometrica

Quando campioniamo una sinusoide troppo lentamente rispetto alla sua frequenza, i punti campionati possono "sembrare" appartenere a una sinusoide diversa, a frequenza più bassa, che passa esattamente per gli stessi punti campionati. Il campionatore non ha modo di distinguere le due situazioni: **infinite sinusoidi diverse producono la stessa sequenza di campioni** se campionate a una $f_s$ insufficiente. Questo è il motivo per cui l'informazione persa nell'aliasing non è recuperabile a posteriori.

### Formula della frequenza di alias

$$f_{alias} = \left| f - k \cdot f_s \right|$$

dove $k$ è l'intero che minimizza il valore assoluto (cioè porta $f_{alias}$ nell'intervallo $[0, f_s/2]$). In pratica, $k = \text{round}(f / f_s)$.

### Come simuliamo l'undersampling in questo notebook

Poiché in questo laboratorio generiamo i segnali digitalmente (non partiamo da un segnale fisico continuo), simuliamo il campionamento "a bassa frequenza" in due passi:

1. **Decimazione:** partiamo da un segnale già campionato ad alta risoluzione ($f_s$ originale, 44100 Hz) e ne teniamo solo un campione ogni $N$, ottenendo un segnale effettivamente campionato a $f_s / N$.
2. **Ricostruzione per l'ascolto:** per poter riprodurre il risultato con `sounddevice` (che si aspetta una frequenza di riproduzione fissa), "ripetiamo" ogni campione tenuto $N$ volte (tecnica chiamata *zero-order hold*), riportando il segnale alla frequenza di riproduzione originale. Questo introduce anche una lieve distorsione "a gradini" aggiuntiva, che è un artefatto della tecnica di ricostruzione scelta (la più semplice possibile), non del fenomeno di aliasing in sé — è importante distinguere concettualmente le due cose.

### Approfondimento: cos'è esattamente lo zero-order hold

Lo **zero-order hold** (ZOH, letteralmente "mantenimento di ordine zero") è la tecnica di ricostruzione più semplice possibile per passare da una sequenza di campioni discreti a un segnale continuo (o, come nel nostro caso, a un segnale campionato a una frequenza più alta ma di forma compatibile).

**Idea:** ogni campione viene "tenuto costante" (da cui *hold*) per tutta la durata del suo intervallo di campionamento, fino all'arrivo del campione successivo. Il risultato, visivamente, è una forma "a gradini" (o *staircase*): il segnale rimane piatto per un intervallo, poi salta istantaneamente al valore del campione successivo, invece di variare in modo continuo come il segnale originale.

Questo è esattamente ciò che fa la funzione `ricostruisci_per_ascolto` implementata più sotto: la funzione `np.repeat` ripete letteralmente ogni valore campionato "fattore" volte, producendo proprio quella forma a gradini.

**Perché lo usiamo in questo notebook:** è la tecnica di ricostruzione più semplice da implementare, e ci serve solo per un motivo pratico — riportare il segnale sotto-campionato a una frequenza compatibile con `sounddevice`, così da poterlo ascoltare. Non è pensata per essere una ricostruzione "di qualità": esistono tecniche molto migliori (si veda sotto), ma lo ZOH è sufficiente per il nostro scopo didattico, che è illustrare l'aliasing, non ottimizzare la fedeltà della ricostruzione.

**Cosa introduce di indesiderato (oltre all'aliasing):** i "gradini" bruschi generati dallo ZOH corrispondono, nel dominio della frequenza, all'introduzione di componenti spettrali aggiuntive ad alta frequenza (armoniche indesiderate), diverse e distinte dall'aliasing vero e proprio. In altre parole: anche se il campionamento fosse stato eseguito correttamente (sopra la soglia di Nyquist), la ricostruzione con ZOH introdurrebbe comunque una leggera colorazione del suono, percepibile come un timbro leggermente più "metallico" o "squadrato" rispetto al segnale originale. È per questo motivo che nell'Esperimento 1 confrontiamo lo spettro originale con quello ricostruito: parte della discrepanza osservata, specialmente alle frequenze più alte, può essere dovuta allo ZOH e non solo all'aliasing.

**Come si risolverebbe in un sistema reale:** nei convertitori digitale-analogico (DAC) reali, l'uscita a gradini dello ZOH viene fatta passare attraverso un **filtro di ricostruzione** (un filtro passa-basso analogico, concettualmente simile al filtro anti-aliasing visto in precedenza, ma applicato in uscita anziché in ingresso). Questo filtro "smussa" i gradini, attenuando le componenti ad alta frequenza introdotte dallo ZOH e restituendo un segnale molto più vicino a quello continuo originale. Un'alternativa teoricamente ideale (ma non fisicamente realizzabile in modo esatto) è la **ricostruzione tramite interpolazione sinc**, prevista dal teorema del campionamento stesso: se il campionamento è stato corretto (rispettando Nyquist), sommando funzioni sinc opportunamente pesate e centrate su ciascun campione si otterrebbe una ricostruzione matematicamente perfetta del segnale originale.

| Tecnica di ricostruzione | Qualità | Complessità | Uso in questo notebook |
|---|---|---|---|
| Zero-order hold (gradini) | Bassa (introduce armoniche spurie) | Minima | Sì, per semplicità didattica |
| Filtro di ricostruzione passa-basso | Buona (uso reale nei DAC) | Media | No |
| Interpolazione sinc | Ideale (teoricamente perfetta se Nyquist è rispettato) | Alta (costosa da calcolare) | No |

In [ ]:
def sottocampiona(segnale, fs_originale, fs_target):
    """Sotto-campiona un segnale prendendo un campione ogni N (decimazione).

    Parametri
    ---------
    segnale : np.ndarray
        Segnale di partenza, campionato a fs_originale.
    fs_originale : float
        Frequenza di campionamento del segnale in ingresso.
    fs_target : float
        Frequenza di campionamento desiderata (deve essere <= fs_originale).

    Ritorna
    -------
    np.ndarray
        Segnale decimato, campionato a fs_target.

    NB: fs_originale deve essere un multiplo intero di fs_target per questa
    implementazione semplificata basata su decimazione via slicing.
    """
    fattore = fs_originale / fs_target
    if not fattore.is_integer():
        raise ValueError(
            f"fs_originale ({fs_originale}) deve essere multiplo intero di fs_target ({fs_target})"
        )
    fattore = int(fattore)

    # segnale[::fattore] tiene un campione ogni "fattore" campioni originali:
    # e' l'equivalente discreto del "misurare il segnale meno spesso"
    return segnale[::fattore]


def ricostruisci_per_ascolto(segnale_campionato, fattore):
    """Ricostruzione 'a scalini' (zero-order hold).

    Riporta il segnale sotto-campionato alla frequenza di riproduzione
    originale ripetendo ogni campione "fattore" volte, cosi' da poterlo
    ascoltare con sounddevice (che riproduce sempre a fs_originale in
    questo notebook). Questa NON e' una ricostruzione ideale (che
    richiederebbe un filtro passa-basso / interpolazione sinc), mostra
    solo, in modo grezzo, cosa "sente" un sistema dopo il sottocampionamento.
    """
    return np.repeat(segnale_campionato, fattore)


def calcola_frequenza_alias(f_segnale, fs_campionamento):
    """Calcola la frequenza di alias attesa teoricamente.

    Implementa la formula f_alias = |f - k*fs|, con k intero scelto in modo
    da minimizzare il risultato (equivalente a "ripiegare" f nell'intervallo
    [0, fs/2], la banda rappresentabile a questa frequenza di campionamento).
    """
    k = round(f_segnale / fs_campionamento)
    f_alias = abs(f_segnale - k * fs_campionamento)
    return f_alias


def plot_spettro(sig, fs, titolo="", xlim=5000):
    """Visualizza lo spettro di ampiezza (FFT) di un segnale.

    Usiamo la FFT (Fast Fourier Transform) per passare dal dominio del tempo
    (ampiezza in funzione del tempo) al dominio della frequenza (ampiezza in
    funzione della frequenza): e' lo strumento che ci permette di "vedere"
    a quale frequenza si trova davvero l'energia del segnale, incluso
    l'eventuale spostamento dovuto all'aliasing.
    """
    n = len(sig)

    # np.fft.rfft calcola la FFT per segnali reali (piu' efficiente di fft
    # completa, dato che lo spettro di un segnale reale e' simmetrico).
    # rfftfreq calcola i valori di frequenza (asse x) corrispondenti.
    freqs = np.fft.rfftfreq(n, d=1/fs)
    spettro = np.abs(np.fft.rfft(sig)) / n  # normalizziamo per confrontare ampiezze

    plt.figure(figsize=(8, 3))
    plt.plot(freqs, spettro)
    plt.xlabel("Frequenza (Hz)")
    plt.ylabel("Ampiezza normalizzata")
    plt.title(titolo)
    plt.xlim(0, xlim)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return freqs, spettro

### Esperimento 1: segnale a 1000 Hz, campionato a 900 Hz

Nyquist richiederebbe $f_s \geq 2000$ Hz. Campionando a 900 Hz (ben sotto soglia) ci aspettiamo un aliasing marcato: proviamo a calcolare a mano la frequenza attesa prima di eseguire il codice.

**Calcolo preliminare (da fare prima di eseguire la cella):** con $f = 1000$ Hz e $f_s = 900$ Hz, quale valore di $k$ minimizza $|f - k \cdot f_s|$? Quale frequenza di alias prevedete?

In [ ]:
# Parametri dell'esperimento
f_segnale = 1000
fs_originale = 44100
fs_sottocampionamento = 900  # deve dividere fs_originale esattamente per questa demo

# fs_originale non e' un multiplo esatto di 900, quindi calcoliamo il fattore
# di decimazione intero piu' vicino e la frequenza effettiva risultante
fattore = round(fs_originale / fs_sottocampionamento)
fs_effettiva = fs_originale / fattore
print(f"Frequenza di campionamento effettiva usata: {fs_effettiva:.1f} Hz (fattore di decimazione: {fattore})")

# Calcoliamo la previsione teorica PRIMA di generare il segnale ricostruito,
# cosi' da poterla confrontare "alla cieca" con l'osservazione sperimentale
f_alias_atteso = calcola_frequenza_alias(f_segnale, fs_effettiva)
print(f"Frequenza di alias attesa (teorica): {f_alias_atteso:.1f} Hz")

# 1. Generiamo il segnale originale ad alta risoluzione (nessun aliasing qui)
segnale_orig, t_orig = genera_tono(f_segnale, durata=2.0, fs=fs_originale)

# 2. Sotto-campioniamo: simuliamo la misurazione del segnale a fs_effettiva
segnale_sotto = sottocampiona(segnale_orig, fs_originale, fs_effettiva)

# 3. Ricostruiamo (zero-order hold) per poter ascoltare il risultato
segnale_ricostruito = ricostruisci_per_ascolto(segnale_sotto, fattore)

# Confronto grafico nel dominio del tempo: originale vs ricostruito (primi 10 ms)
plot_forma_onda(segnale_orig, t_orig, fs_originale, n_ms=10,
                 titolo=f"Segnale originale a {f_segnale} Hz")
plot_forma_onda(segnale_ricostruito, t_orig, fs_originale, n_ms=10,
                 titolo=f"Segnale ricostruito dopo sotto-campionamento a {fs_effettiva:.0f} Hz")

# Confronto nel dominio della frequenza: qui si vede chiaramente lo spostamento
# del picco dovuto all'aliasing
plot_spettro(segnale_orig, fs_originale, titolo="Spettro - segnale originale")
plot_spettro(segnale_ricostruito, fs_originale, titolo="Spettro - segnale ricostruito (con alias)")

In [ ]:
print("Ascolto: segnale originale a", f_segnale, "Hz")
sd.play(segnale_orig, samplerate=fs_originale)
sd.wait()

print("Ascolto: segnale ricostruito dopo sotto-campionamento (aliasing)")
sd.play(segnale_ricostruito, samplerate=fs_originale)
sd.wait()

# Domanda: i due suoni hanno la stessa altezza (pitch) percepita?
# Il pitch percepito dovrebbe corrispondere alla frequenza di alias, non a f_segnale.

**Verifica sperimentale:** confrontate il picco osservato nello spettro del segnale ricostruito con il valore di `f_alias_atteso` calcolato sopra. Corrispondono? Se non corrispondono esattamente, discutete possibili cause (risoluzione in frequenza della FFT, effetti di finestratura, durata del segnale).

> **Approfondimento — risoluzione in frequenza della FFT:** la FFT calcolata su un segnale di durata $T$ secondi ha una risoluzione in frequenza pari a $\Delta f = 1/T$. Con `durata=2.0` secondi, la risoluzione è di 0.5 Hz: piccoli scostamenti dal valore teorico entro questo ordine di grandezza sono normali e non indicano un errore concettuale.

### Esercizio 3.1 (livello intermedio)

Ripetete l'esperimento con $f = 1000$ Hz ma provando le seguenti frequenze di campionamento (verificando che siano divisori interi di 44100, altrimenti scegliete il divisore più vicino): 1050 Hz, 1470 Hz, 2205 Hz. Per ciascuna:
1. Calcolate a mano la frequenza di alias attesa.
2. Eseguite il codice e confrontate con l'osservazione.
3. Man mano che $f_s$ si avvicina a $2f = 2000$ Hz, cosa succede alla frequenza di alias?

### Esercizio 3.2 (livello intermedio)

Scrivete una funzione `verifica_regime(f_segnale, fs)` che restituisca una stringa tra `"undersampling"`, `"limite"` o `"corretto"` a seconda che $f_s$ sia minore, uguale o maggiore del doppio di `f_segnale` (con una tolleranza a scelta per il caso "limite", ad esempio ±1%). Usatela per classificare automaticamente i casi analizzati finora.

## 4. Campionamento alla soglia di Nyquist e sopra

Ripetiamo l'esperimento con:
- $f_s = 2f$ esatto (caso limite — spesso instabile/ambiguo a seconda della fase)
- $f_s \gg 2f$ (campionamento corretto, es. 44100 Hz)

### Perché il caso $f_s = 2f$ è un caso limite fragile

Quando $f_s = 2f$ esatto, si campionano esattamente **due punti per ogni periodo** dell'onda. Se questi due punti cadono esattamente sugli zeri della sinusoide (cosa che dipende dalla fase iniziale $\varphi$), il segnale campionato risulta **identicamente nullo**, pur essendo il segnale originale tutt'altro che silenzioso! Questo è un esempio concreto del perché la teoria richiede $f_s \geq 2f$ ma la pratica ingegneristica preferisce sempre un margine ($f_s > 2f$).

In [ ]:
def esperimento_campionamento(f_segnale, fs_target, fs_originale=44100, durata=2.0, ascolta=True, fase=0.0):
    """Esegue l'intera pipeline campionamento -> ricostruzione -> visualizzazione
    per una data combinazione di frequenza del segnale e frequenza di campionamento.

    E' la funzione "di alto livello" che useremo per gli esperimenti successivi,
    per evitare di ripetere ogni volta gli stessi passaggi (decimazione,
    ricostruzione, plotting).
    """
    # Il fattore di decimazione deve essere intero: approssimiamo se necessario
    fattore = round(fs_originale / fs_target)
    fs_effettiva = fs_originale / fattore

    # Generiamo il segnale ad alta risoluzione, con la fase specificata
    segnale_orig, t_orig = genera_tono(f_segnale, durata=durata, fs=fs_originale, fase=fase)

    # Sotto-campioniamo alla frequenza effettiva e ricostruiamo per l'ascolto
    segnale_camp = sottocampiona(segnale_orig, fs_originale, fs_effettiva)
    segnale_ricostruito = ricostruisci_per_ascolto(segnale_camp, fattore)

    print(f"Segnale: {f_segnale} Hz | Campionamento: {fs_effettiva:.1f} Hz "
          f"| Nyquist richiederebbe: {2*f_segnale} Hz | fase iniziale: {fase:.3f} rad")

    plot_spettro(segnale_ricostruito, fs_originale,
                 titolo=f"Spettro - f={f_segnale} Hz campionato a {fs_effettiva:.0f} Hz")

    if ascolta:
        sd.play(segnale_ricostruito, samplerate=fs_originale)
        sd.wait()

    return segnale_ricostruito


# Caso limite: fs = 2f esatto, fase iniziale nulla
esperimento_campionamento(f_segnale=1000, fs_target=2000, fase=0.0)

# Caso corretto: fs >> 2f (ampio margine rispetto alla soglia)
esperimento_campionamento(f_segnale=1000, fs_target=44100)

**Osservazione da verificare in aula:** cosa succede esattamente al caso $f_s = 2f$ con fase nulla? Con `fase=0.0`, i campioni cadono esattamente sugli zeri della sinusoide: il segnale ricostruito rischia di risultare (quasi) silenzioso.

Proviamo ora a ripetere lo stesso esperimento cambiando **solo la fase iniziale**, mantenendo $f_s = 2f$: questo isola l'effetto della fase da quello della frequenza di campionamento.

In [ ]:
# Confrontiamo diverse fasi iniziali nel caso limite fs = 2f, per isolare
# l'effetto della fase (che nella teoria "ideale" non dovrebbe influenzare
# la rappresentabilita' del segnale, ma nella pratica del campionamento discreto sì)
fasi_da_provare = [0.0, np.pi/4, np.pi/2]

for fase in fasi_da_provare:
    print(f"--- Fase iniziale: {fase:.3f} rad ---")
    esperimento_campionamento(f_segnale=1000, fs_target=2000, fase=fase, ascolta=True)

### Esercizio 4.1 (livello intermedio)

Con $f_s = 2f$ esatto, per quale valore di fase (tra 0 e $\pi/2$) l'ampiezza del segnale campionato è **massima**? Verificate la vostra ipotesi modificando il parametro `fase` nella cella precedente.

### Esercizio 4.2 (livello avanzato)

Scrivete un piccolo script che, fissati $f = 1000$ Hz e $f_s = 2000$ Hz, calcoli (senza riprodurre l'audio) l'ampiezza massima del segnale campionato al variare della fase tra 0 e $2\pi$ (ad esempio su 20 valori equispaziati), e produca un grafico "ampiezza campionata vs fase". Cosa osservate?

*Suggerimento:* potete riusare `sottocampiona` e `genera_tono` con `ascolta=False`, calcolando `np.max(np.abs(segnale_camp))` per ciascuna fase.

## 5. Esperimento guidato: previsione e verifica

Lavorate in coppia. Seguite questi passi:

1. Scegliete una frequenza del segnale $f$ (Hz).
2. Scegliete una frequenza di campionamento $f_s$ (Hz), tale che `fs_originale` (44100 Hz) sia un multiplo intero di $f_s$ (altrimenti il codice userà il divisore intero più vicino).
3. **Prima di eseguire il codice**, prevedete: ci sarà aliasing? Se sì, a quale frequenza?
4. Eseguite la cella sottostante e confrontate la vostra previsione con il risultato osservato (grafico e ascolto).
5. Ripetete con almeno altre 2 combinazioni di $f$ e $f_s$, compilando la tabella riassuntiva.

In [ ]:
# --- Modificate questi valori ---
f_scelta = 1200       # Hz - frequenza del segnale
fs_scelta = 1600      # Hz - frequenza di campionamento
# --------------------------------

# 1. Previsione teorica, calcolata PRIMA di generare/ascoltare il segnale
f_alias_previsto = calcola_frequenza_alias(f_scelta, fs_scelta)
print(f"Previsione teorica di frequenza di alias: {f_alias_previsto:.1f} Hz")
print(f"(Nyquist per f={f_scelta} Hz richiederebbe fs >= {2*f_scelta} Hz)")

# 2. Esecuzione dell'esperimento e confronto con l'osservazione
risultato = esperimento_campionamento(f_segnale=f_scelta, fs_target=fs_scelta)

**Tabella da compilare (su carta o in una cella markdown):**

| Tentativo | f (Hz) | fs (Hz) | Previsione alias (Hz) | Alias osservato (Hz) | Corrisponde? |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

### Esercizio 5.1 (livello avanzato)

Scegliete una coppia $(f, f_s)$ con $f_s < 2f$ tale per cui la frequenza di alias risulti **udibile ma vicina agli estremi dell'intervallo udibile** (ad esempio vicina a 20 Hz o a 18-20 kHz). Cosa cambia nella percezione soggettiva rispetto ai casi precedenti?

### Esercizio 5.2 (livello avanzato — sfida)

Fissata $f_s = 8000$ Hz (frequenza tipica della telefonia analogica), trovate almeno **due frequenze diverse** di segnale, $f_1$ e $f_2$, che producano la **stessa identica** frequenza di alias. Verificate sperimentalmente che i due segnali ricostruiti risultino indistinguibili nello spettro (a meno di rumore numerico). Questo dimostra concretamente che l'aliasing è un fenomeno **non invertibile**: una volta che si è verificato, non è possibile risalire con certezza al segnale originale a partire dai soli campioni.

## Esercizi di riepilogo (da svolgere a casa / come consegna)

1. **(Base)** Spiegate a parole vostre, senza formule, perché campionare "troppo lentamente" un suono acuto può farlo sembrare più grave.
2. **(Intermedio)** Dato un segnale a 3000 Hz campionato a 4000 Hz, calcolate la frequenza di alias attesa e verificatela con il codice del notebook.
3. **(Intermedio)** Modificate `genera_tono` per generare non un singolo tono puro, ma la somma di due toni puri a frequenze diverse (es. 500 Hz e 3000 Hz). Campionate il segnale risultante a una frequenza insufficiente per la componente a 3000 Hz ma sufficiente per quella a 500 Hz. Cosa osservate nello spettro? Quale componente subisce aliasing e quale no?
4. **(Avanzato)** Implementate un semplice filtro anti-aliasing usando `scipy.signal.butter` e `scipy.signal.lfilter` (filtro passa-basso) da applicare al segnale **prima** della decimazione. Confrontate lo spettro del segnale sotto-campionato con e senza filtro anti-aliasing preliminare: il filtro elimina davvero l'aliasing, o semplicemente lo attenua? Perché?